# 说明

这个单元就是把一些讲过的内容用Microsoft Agent Framework 框架实现一遍，Microsoft Agent Framework 是微软将AutoGen 和 Semantic Kernel (SK)这两个框架的主要优势合并，形成了一个统一的开源框架

## 两个顺序代理：

1. **前台代理**：提供城市的初步吸引力推荐  
2. **礼宾代理**：根据受欢迎程度审查并评分前台代理的推荐  

## 顺序编排的主要优势：

- **迭代优化**：第二个代理改进第一个代理的工作  
- **专业化**：每个代理在流程中都有特定的角色  
- **质量控制**：内置的审查和验证步骤  
- **信息流清晰**：代理之间结构化的交接  

## 前提条件：
- 安装 Microsoft Agent Framework  
- 配置 DashScope API key 或其他 OpenAI 兼容服务  
- 了解基本代理概念  

注意：如果你使用 `qwen-max`，建议使用 `OpenAIChatCompletionClient`，因为它走 `chat.completions` 接口，比 `OpenAIChatClient` 的 `responses` 接口更兼容 DashScope。


In [2]:
import sys
!{sys.executable} -m pip install agent-framework-core agent-framework-openai agent-framework-orchestrations


  Using cached agent_framework_core-1.2.2-py3-none-any.whl.metadata (10 kB)
Using cached agent_framework_core-1.2.2-py3-none-any.whl (348 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [agent-framework-openai]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [15]:
# ===== 第一部分：导入必要的库 =====
# 异步编程支持（使代码能同时处理多任务）
import asyncio
# JSON数据处理
import json
# 操作系统功能（如环境变量）
import os
# 类型提示（帮助理解代码）
from typing import Any, cast

# Microsoft Agent Framework 当前版本核心组件
# 注意：新版本里 ChatMessage 改名为 Message，
# SequentialBuilder 位于 agent_framework.orchestrations 命名空间下，
# agent 需要通过 Agent(...) 显式创建。
from agent_framework import Agent, Message
from agent_framework.orchestrations import SequentialBuilder

# OpenAI 兼容客户端
# 这里使用 OpenAIChatCompletionClient，以兼容 DashScope 的 qwen-max。
from agent_framework.openai import OpenAIChatCompletionClient
# 从.env文件加载环境变量
from dotenv import load_dotenv
# 在Jupyter中显示HTML内容
from IPython.display import HTML, display
# 用于定义结构化数据模型
from pydantic import BaseModel, Field

print("All imports successful!")


All imports successful!


## 第一步：定义用于结构化输出的 Pydantic 模型

这些模型定义了每个代理将返回的模式。前台代理提供推荐，礼宾代理提供评论和评分。


In [16]:
# ===== 第二部分：定义结构化输出模型 =====
# 这些模型确保代理返回的数据格式正确、完整
# 注意：为了提高 qwen-max 在结构化输出下的稳定性，
# 对少量容易偶发缺失的字段设置默认值，避免一次漏字段就导致整个工作流中断。
class AttractionRecommendation(BaseModel):
    """Attraction recommendation from the front desk agent.
    前台代理返回的景点推荐数据结构"""

    city: str  # 城市名称
    attraction_name: str  # 景点名称
    description: str  # 景点描述
    category: str  # 景点类别（如"博物馆"、"地标"等）
    recommended_duration: str  # 推荐游览时长（如"2-3小时"）
    why_recommended: str = Field(default="No reason provided.")  # 推荐理由
    best_time_to_visit: str = Field(default="Best time not specified.")  # 最佳参观时间


class AttractionReview(BaseModel):
    """Expert review and rating from the concierge agent.
    礼宾代理返回的景点评审数据结构"""

    attraction_name: str  # 景点名称
    city: str  # 城市名称
    popularity_score: int = Field(default=7)  # 流行度评分（1-10分）
    popularity_reasoning: str = Field(default="No popularity reasoning provided.")  # 评分理由
    visitor_rating: float = Field(default=4.0)  # 游客评分（1.0-5.0分）
    pros: list[str] = Field(default_factory=list)  # 优点列表
    cons: list[str] = Field(default_factory=list)  # 缺点列表
    concierge_recommendation: str = Field(default="No concierge recommendation provided.")  # 礼宾建议
    alternative_suggestions: list[str] = Field(default_factory=list)  # 替代建议


## 第2步：加载环境变量

下面使用 OpenAI 兼容方式配置聊天客户端。

如果你使用的是：

- GitHub Models：可以继续填写 `GITHUB_ENDPOINT / GITHUB_TOKEN`
- DashScope + qwen-max：建议填写 `DASHSCOPE_API_KEY`，并把 `base_url` 指向兼容接口

本 notebook 当前按 `DashScope + qwen-max` 方式演示。


In [17]:
# ===== 第三部分：配置LLM服务 =====

# Load environment variables
# 加载环境变量（从.env文件）
load_dotenv()

# 创建聊天客户端 - 这是连接到AI模型的桥梁
# 这里使用 chat.completions 风格客户端，以兼容 DashScope 的 qwen-max
# chat_client = OpenAIChatCompletionClient(
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # DashScope OpenAI兼容接口
#     api_key=os.environ.get("DASHSCOPE_API_KEY"),                  # DashScope API Key
#     model="qwen-max"                                              # 使用的模型名称
# )
chat_client = OpenAIChatCompletionClient(
    base_url="https://models.inference.ai.azure.com/",  # DashScope OpenAI兼容接口
    api_key=os.environ.get("GITHUB_TOKEN"),                  # DashScope API Key
    model="gpt-4o-mini"                                              # 使用的模型名称
)

print("Chat client configured successfully!")


Chat client configured successfully!


## 第三步：创建两个顺序代理

每个代理在顺序工作流程中都有特定的角色。前台代理负责提供推荐，而礼宾代理负责审核并评分这些推荐。


In [18]:
# ===== 第四部分：创建专业代理 =====
# 代理1：前台接待员（提供景点初步推荐）
# 指令中文翻译如下：
# 你是酒店前台工作人员，专精于当地景点推荐。
# 当客人询问城市景点时，提供一个经过充分研究的推荐，包括景点特色、推荐游览时长和最佳参观时间。
# 保持热情友好的态度，返回指定格式的JSON数据。
# Agent 1: Front Desk Agent (Makes initial recommendations)
front_desk_agent = Agent(
    client=chat_client,
    instructions=(
        "You are a knowledgeable hotel front desk agent who specializes in local attractions. "
        "When a guest asks about attractions in a city, provide a single, well-researched recommendation "
        "for a popular tourist attraction. Focus on giving practical information including what makes "
        "this attraction special, how long to spend there, and the best time to visit. "
        "Be helpful and enthusiastic about your recommendation. "
        "You MUST return valid JSON only. "
        "Always include all of these fields exactly: "
        "city, attraction_name, description, category, recommended_duration, why_recommended, best_time_to_visit."
    ),
    name="front_desk_agent",
    default_options={"response_format": AttractionRecommendation},
)

# Agent 2: Concierge Agent (Reviews and rates recommendations)
# 代理2：礼宾（提供专家评审和评分）
# 指令中文翻译如下：
# 你是一个专家级礼宾，精通全球旅游景点。
# 你将收到一个景点推荐，并需要提供专家评审和评分。
# 根据景点的流行度、游客满意度和整体质量进行评估。  
# 提供流行度评分（1-10分）、游客评分（1.0-5.0分）、列出优缺点，并给出专业评估。
# 如有必要，还要建议替代景点。返回指定格式的JSON数据。
concierge_agent = Agent(
    client=chat_client,
    instructions=(
        "You are an expert concierge with extensive knowledge of tourist attractions worldwide. "
        "You will receive an attraction recommendation and must provide an expert review and rating. "
        "Evaluate the recommendation based on the attraction's popularity, visitor satisfaction, "
        "and overall quality. Provide a popularity score (1-10), visitor rating (1.0-5.0), "
        "list pros and cons, and give your professional assessment. "
        "Also suggest alternative attractions if appropriate. "
        "You MUST return valid JSON only. "
        "Always include all of these fields exactly: "
        "attraction_name, city, popularity_score, popularity_reasoning, visitor_rating, pros, cons, concierge_recommendation, alternative_suggestions."
    ),
    name="concierge_agent",
    default_options={"response_format": AttractionReview},
)


## 第四步：构建顺序工作流

SequentialBuilder 创建了一个工作流，其中：
1. **前台接待员** 接收用户输入并提供推荐
2. **礼宾员** 接收前台的推荐并进行专家审查
3. **输出** 包含原始推荐和专家审查


In [19]:
# ===== 第五部分：构建顺序工作流 =====
# 使用SequentialBuilder创建代理执行流程
# # Build the sequential workflow using SequentialBuilder
workflow = SequentialBuilder(
    participants=[front_desk_agent, concierge_agent]  # 指定参与的代理及顺序
).build() # 构建工作流


display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Sequential Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Flow:</strong><br>
        • User Input → <strong>Front Desk Agent</strong> (recommendation)<br>
        • Front Desk Output → <strong>Concierge Agent</strong> (review & rating)<br>
        • Final Output → Combined recommendation + expert review
    </p>
</div>
"""))

In [20]:
# ===== 第六部分：定义结果展示函数 =====
async def display_attraction_recommendation(city: str):
    """Run the sequential workflow and display formatted results.
    运行工作流并显示格式化结果"""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>正在处理 {city} 的景点推荐</h3>
        <p style='margin: 0;'><strong>状态：</strong>运行顺序工作流中...</p>
    </div>
    """))

    # 执行工作流（用户提问 → 前台代理 → 礼宾代理）
    events = await workflow.run(f"I want to visit an attraction in {city}")
    outputs = events.get_outputs()

    if outputs:
        # 当前版本里 outputs[0] 是 AgentResponse，不是直接的消息列表。
        response = outputs[0]
        # 真正的消息列表位于 response.messages 中。
        messages: list[Message] = response.messages

        # Find front desk and concierge responses
        front_desk_response = None
        concierge_response = None

        for msg in messages:
            if msg.author_name == "front_desk_agent":
                front_desk_response = msg.text
            elif msg.author_name == "concierge_agent":
                concierge_response = msg.text

        display(HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0;'>推荐景点 {city}</h2>
            <p style='margin: 0; font-size: 14px; opacity: 0.9;'>由顺序代理工作流生成</p>
        </div>
        """))

        if front_desk_response:
            try:
                recommendation_data = AttractionRecommendation.model_validate_json(front_desk_response)
                display_front_desk_section(recommendation_data)
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>前台响应错误解析:</strong> {str(e)}
                    <details><summary>原始响应</summary>{front_desk_response}</details>
                </div>
                 """))

        if concierge_response:
            try:
                review_data = AttractionReview.model_validate_json(concierge_response)
                display_concierge_section(review_data)
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>Error parsing concierge response:</strong> {str(e)}
                    <details><summary>Raw response</summary>{concierge_response}</details>
                </div>
                """))


def display_front_desk_section(data: AttractionRecommendation):
    """Display front desk recommendation in a formatted section.
    以美观格式显示前台推荐"""

    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏨 Front Desk Recommendation</h3>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>{data.attraction_name}</h4>
            <span style='background: #2196f3; color: white; padding: 4px 8px; border-radius: 12px; font-size: 12px;'>{data.category}</span>
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Description:</strong> {data.description}
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Why Recommended:</strong> {data.why_recommended}
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Recommended Duration:</strong> {data.recommended_duration}
        </div>
        <div>
            <strong style='color: #333;'>Best Time to Visit:</strong> {data.best_time_to_visit}
        </div>
    </div>
    """))


def display_concierge_section(data: AttractionReview):
    """Display concierge review in a formatted section.
    以美观格式显示礼宾评审"""

    star_rating = "⭐" * int(data.visitor_rating) + "☆" * (5 - int(data.visitor_rating))
    popularity_bar = "🟩" * data.popularity_score + "⬜" * (10 - data.popularity_score)
    pros_list = "".join([f"<li style='color: #4caf50;'>✓ {pro}</li>" for pro in data.pros])
    cons_list = "".join([f"<li style='color: #f44336;'>✗ {con}</li>" for con in data.cons])
    alternatives_list = "".join([f"<li>{alt}</li>" for alt in data.alternative_suggestions])

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🎩 Concierge Expert Review</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Popularity Score:</strong> {data.popularity_score}/10 {popularity_bar}
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Visitor Rating:</strong> {data.visitor_rating}/5 {star_rating}
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Popularity Reasoning:</strong> {data.popularity_reasoning}
        </div>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Concierge Recommendation:</strong> {data.concierge_recommendation}
        </div>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px;'>
            <div>
                <strong style='color: #333;'>Pros:</strong>
                <ul style='margin: 8px 0;'>{pros_list}</ul>
            </div>
            <div>
                <strong style='color: #333;'>Cons:</strong>
                <ul style='margin: 8px 0;'>{cons_list}</ul>
            </div>
        </div>
        <div style='margin-top: 15px;'>
            <strong style='color: #333;'>Alternative Suggestions:</strong>
            <ul style='margin: 8px 0;'>{alternatives_list}</ul>
        </div>
    </div>
    """))


# Test with Stockholm
await display_attraction_recommendation("Stockholm")


## 第8步：工作流程分析 - 理解顺序流程

让我们来研究信息如何在代理之间流动，并分析对话历史记录。


In [ ]:
async def analyze_sequential_flow(city: str):
    """Analyze the sequential flow between agents."""

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>Sequential Flow Analysis for {city}</h3>
        <p style='margin: 0;'>Examining agent interactions and information handoff...</p>
    </div>
    """))

    # Run the workflow
    events = await workflow.run(f"I want to visit an attraction in {city}")
    outputs = events.get_outputs()

    if outputs:
        # 当前版本里 outputs[0] 是 AgentResponse，不是直接可迭代的消息列表。
        response = outputs[0]
        messages: list[Message] = response.messages

        display(HTML(f"""
        <div style='padding: 25px; background: #f3e5f5; border-radius: 12px; margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0; color: #7b1fa2;'>Conversation Flow Analysis</h2>
        </div>
        """))

        # Display each message in the sequence
        for i, msg in enumerate(messages, 1):
            role_color = {
                "user": "#2196f3",
                "front_desk_agent": "#4caf50",
                "concierge_agent": "#ff9800"
            }.get(msg.author_name or "user", "#666666")

            role_name = {
                "user": "👤 User",
                "front_desk_agent": "🏨 Front Desk Agent",
                "concierge_agent": "🎩 Concierge Agent"
            }.get(msg.author_name or "user", "Unknown")

            content_preview = msg.text[:200] + "..." if len(msg.text) > 200 else msg.text
            display(HTML(f"""
            <div style='padding: 15px; background: white; border-left: 4px solid {role_color}; border-radius: 4px; margin: 10px 0; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                <div style='display: flex; align-items: center; margin-bottom: 10px;'>
                    <span style='font-weight: bold; color: {role_color}; margin-right: 10px;'>Step {i}:</span>
                    <span style='font-weight: bold; color: {role_color};'>{role_name}</span>
                </div>
                <div style='color: #555; font-size: 14px; line-height: 1.4;'>
                    {content_preview}
                </div>
            </div>
            """))

        # Analyze the flow
        display(HTML(f"""
        <div style='padding: 20px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 8px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>Flow Analysis Summary</h3>
            <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
                <li><strong>Total Messages:</strong> {len(messages)}</li>
                <li><strong>Agents Involved:</strong> 2 (Front Desk + Concierge)</li>
                <li><strong>Flow Pattern:</strong> Linear sequential (User → Agent 1 → Agent 2)</li>
                <li><strong>Information Handoff:</strong> Front desk recommendation becomes concierge input</li>
                <li><strong>Output Quality:</strong> Enhanced through expert review and rating</li>
            </ul>
        </div>
        """))


# Analyze the flow for Barcelona
await analyze_sequential_flow("Barcelona")


concierge_agent {"attraction_name":"Park Güell","city":"Barcelona","popularity_score":8,"popularity_reasoning":"Park Güell is one of the most famous parks in Barcelona, drawing around 4 million visitors each year. It



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
